# Bureau d'Analyse Terrestre — Détection de canulars Klaxo-3

Notebook conforme au **Manuel du Bureau** (12 sections).
Objectif : détecter les relevés marqués comme canulars (`hoax`) à partir des transmissions UFO NUFORC.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import pandas as pd

plt.style.use("seaborn-v0_8-whitegrid")
DATA_PATH = ROOT / "data" / "releves_klaxo3.csv"

## 1. Charger un fichier hostile

Lecture ligne par ligne avec `csv.reader` — aucune perte silencieuse.

In [ ]:
from src.load_data import load_report

report = load_report(DATA_PATH)
df = report["df"]
print(f"Gardées : {report['n_gardees']:,} | Écartées : {report['n_ecartees']:,}")

## 2. Coercion de types

`errors='coerce'` convertit les valeurs fautives en `NaN` au lieu de basculer toute la colonne en texte.

In [ ]:
from src.clean import coercion_report

coercion_report(df)

## 3. Fabriquer une cible — weak supervision

La cible `canular` est une **opinion** (règle `contains('hoax')`), pas une mesure de vérité.

In [ ]:
from src.clean import prepare_features

df = prepare_features(df)
print(f"Canulars : {df['canular'].sum()} ({100 * df['canular'].mean():.2f}%)")

## 5. Fuite de données

Retrait des notes Bureau `((...))` — seul le témoignage terrain (`temoin`) entre dans le modèle.

In [ ]:
from src.clean import leakage_report

leakage_report(df)

## 7. Découper honnêtement

Split **temporel** sur `date_posted` — le modèle ne doit pas lire l'avenir.

In [ ]:
from src.split import label_shift_report, temporal_split, yearly_canular_rate
from src.pipeline import get_X

i_train, i_test = temporal_split(df)
label_shift_report(df, i_train, i_test)

yearly = yearly_canular_rate(df)
yearly.plot(x="annee", y="taux_canular", figsize=(10, 4), legend=False)
plt.title("Taux de canulars par année de publication")
plt.ylabel("Taux")
plt.show()

## 8. Les trous ne sont pas des accidents

Un relevé troué a plus de chances d'être un canular — signal MNAR.

In [ ]:
from src.features import missing_signal_report

missing_signal_report(df)

## 6. Modèle bête + 9-10. Pipeline scikit-learn

Baseline `DummyClassifier` puis pipeline complète `ColumnTransformer` + `LogisticRegression`.

In [ ]:
from src.evaluate import baseline_report, evaluate_model
from src.pipeline import build_pipeline

X = get_X(df)
y = df["canular"]
X_train, X_test = X.loc[i_train], X.loc[i_test]
y_train, y_test = y.loc[i_train], y.loc[i_test]

baseline_report(X_train, y_train, X_test, y_test)

In [ ]:
pipeline = build_pipeline()
pipeline.fit(X_train, y_train)
y_proba = pipeline.predict_proba(X_test)[:, 1]
y_pred = (y_proba >= 0.5).astype(int)

metrics = evaluate_model(y_test, y_pred, y_proba)
print(f"Precision : {metrics['precision']:.3f}")
print(f"Recall    : {metrics['recall']:.3f}")
print(f"PR-AUC    : {metrics['pr_auc']:.3f}")

## 4. Matrice de confusion

In [ ]:
import numpy as np
import seaborn as sns

cm = np.array(metrics["confusion_matrix"])
fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["Non", "Canular"], yticklabels=["Non", "Canular"], ax=ax)
ax.set_xlabel("Prédit"); ax.set_ylabel("Réel")
plt.show()

## 11. Seuil optimal et calibration

Grille de coûts : 30 crédits/canular raté, 2/fausse alerte.

In [ ]:
from src.evaluate import calibrate_model, optimize_threshold, threshold_comparison

optimize_threshold(y_test, y_proba)
threshold_comparison(y_test, y_proba)

calib = calibrate_model(pipeline, X_train, y_train, X_test, y_test)
print(f"Brier raw : {calib['brier_raw']:.4f} | calibré : {calib['brier_calibrated']:.4f}")

## 12. Explication et audit

Ablation par colonne — le texte porte l'essentiel du signal.

In [ ]:
from src.explain import ablation_by_column, audit_by_country

ablation_by_column(pipeline, X_test, y_test, list(X_test.columns))
audit_by_country(df.loc[i_test], y_test, y_pred).head(10)